<a href="https://colab.research.google.com/github/cylin577/Image2Audio/blob/main/i2agpu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title Install dependency
!pip install gradio
!pip install soundfile
!pip install pillow
!pip install cupy-cuda12x

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.6/50.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.7/56.7 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.8/319.8 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.6/94.6 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.0/78.0 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.4/447.4 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.5/144.5 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.2/73.2 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/1

In [ ]:
import cupy as cp  # For GPU processing
import numpy as np  # For CPU processing
from PIL import Image
import soundfile as sf
import gradio as gr
from scipy.io.wavfile import write
import time  # For time measurement

# English language support
LANGUAGE = {
    "title": "Image-Audio Encoder/Decoder",
    "encode_button": "Select Image to Encode",
    "decode_button": "Select Audio to Decode",
    "success_encode": "Image encoded to audio.",
    "success_decode": "Audio decoded back to image.",
    "error_length": "Decoded data length does not match expected length.",
    "processing_time": "Processing time: {:.6f} seconds",
    "mode_label": "Select Processing Mode (GPU/CPU)",
}

def encode_image_to_audio(image, mode):
    start_time = time.perf_counter()  # Start timing

    # Use CuPy for GPU or NumPy for CPU based on user's choice
    xp = cp if mode == 'GPU(Faster on large image)' else np
    data = xp.array(image)  # Convert image to CuPy or NumPy array
    height, width, _ = data.shape
    audio_data = data / 255.0 * 2 - 1  # Normalize data to [-1, 1]
    audio_data_flattened = audio_data.flatten()  # Flatten the data for audio encoding
    size_info = xp.array([height, width], dtype=xp.int32)  # Store image size
    audio_data_with_size = xp.concatenate([size_info, audio_data_flattened])

    audio_path = "encoded_audio.wav"
    # Save the audio using NumPy array (convert CuPy array if using GPU)
    write(audio_path, 44100, xp.asnumpy(audio_data_with_size).astype(np.float32))

    end_time = time.perf_counter()  # End timing
    processing_time = end_time - start_time  # Calculate elapsed time

    return LANGUAGE["success_encode"] + f" ({LANGUAGE['processing_time'].format(processing_time)})", audio_path

def decode_audio_to_image(audio, mode):
    start_time = time.perf_counter()  # Start timing

    # Read the audio data using soundfile
    audio_data, sample_rate = sf.read(audio)
    height = int(audio_data[0])  # Retrieve height
    width = int(audio_data[1])   # Retrieve width
    # Use CuPy for GPU or NumPy for CPU based on user's choice
    xp = cp if mode == 'GPU(Faster on large image)' else np
    data = xp.array(audio_data[2:])  # Use CuPy/NumPy for data processing
    expected_length = height * width * 3  # Expected data length

    # Check if the length of the data matches the expected length
    if len(data) != expected_length:
        return LANGUAGE["error_length"], None

    # Reconstruct the image data
    img_data = ((data + 1) / 2 * 255).astype(xp.uint8)  # Convert back to image range
    img_data = img_data.reshape((height, width, 3))  # Reshape into an image

    img = Image.fromarray(xp.asnumpy(img_data))  # Convert to NumPy for PIL processing
    image_path = "decoded_image.png"
    img.save(image_path)

    end_time = time.perf_counter()  # End timing
    processing_time = end_time - start_time  # Calculate elapsed time

    return LANGUAGE["success_decode"] + f" ({LANGUAGE['processing_time'].format(processing_time)})", image_path

# Gradio interface setup
def interface():
    with gr.Blocks() as demo:
        gr.Markdown(f"## {LANGUAGE['title']}")

        # Move the GPU/CPU dropdown to the top
        with gr.Row():
            gr.Column(scale=1)  # Empty column for left padding
            with gr.Column(scale=2):  # Center column for the dropdown
                mode_dropdown = gr.Dropdown(choices=["GPU(Faster on large image)", "CPU(Faster on small image)"], label=LANGUAGE["mode_label"], value="GPU(Faster on large image)", interactive=True)
            gr.Column(scale=1)  # Empty column for right padding

        # Encoding and Decoding sections
        with gr.Row():
            with gr.Column():
                gr.Markdown("### Encoding Section")
                img_input = gr.Image(label=LANGUAGE["encode_button"])
                encode_button = gr.Button(LANGUAGE["encode_button"])
            with gr.Column():
                gr.Markdown("### Decoding Section")
                audio_input = gr.Audio(label=LANGUAGE["decode_button"], type="filepath")
                decode_button = gr.Button(LANGUAGE["decode_button"])

        # Outputs for the encoding and decoding sections
        with gr.Row():
            with gr.Column():
                audio_output = gr.File(label="Encoded Audio File")
                encode_button.click(encode_image_to_audio, inputs=[img_input, mode_dropdown], outputs=[gr.Text(label="Outputs"), audio_output])
            with gr.Column():
                img_output = gr.Image(label="Decoded Image")
                decode_button.click(decode_audio_to_image, inputs=[audio_input, mode_dropdown], outputs=[gr.Text(label="Outputs"), img_output])

    return demo

# Run the app with `share=True` and `debug=True`
interface().launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://1ed4854cd4f4384ae1.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
